# Assignment 3 Part C — MNIST experiments with MLflow and DagsHub

This notebook compares four fully connected neural networks using the exact
baseline and the three requested alternatives. Run cells in order using the
project's Python 3.11 kernel. `train.py` contains the same implementation so the
notebook and standalone submission stay consistent.

Before running: follow README.md to install requirements and configure a private
`.env` file. The default below sends all four **50-epoch** runs to DagsHub.
No results are prefilled. A complete run generates real tables, figures and analysis.


In [ ]:
import sys
from pathlib import Path
assert sys.version_info[:2] == (3, 11), 'Select the project Python 3.11 kernel.'
assert Path('train.py').exists(), 'Open this notebook from the project directory.'
import train
from IPython.display import display, Image, Markdown
print('Python:', sys.executable)

TRACKING = 'dagshub'  # 'local' permits offline tracking, but is not DagsHub evidence.
SMOKE = False        # True is diagnostic only: one epoch on a small training subset.
train.configure_runtime()
tracking = train.configure_tracking(TRACKING, SMOKE)
output_dir = train.create_output_dir(SMOKE)
print('Outputs:', output_dir)

## Load and preprocess MNIST

MNIST contains 60,000 original training images and 10,000 separate test images.
Each grayscale image has shape (28, 28). Converting to float32 and dividing by
255 maps pixel intensity to [0, 1]. Integer labels 0–9 work with sparse categorical
cross-entropy; one-hot encoding is unnecessary.

A seed-42 permutation reserves 6,000 images from the original training set for
validation, leaving 54,000 for training. Every experiment uses these same arrays.
The original 10,000 test images are held aside until the winning model is selected.
The test set is loaded here for convenience, but never passed to fit or selection.


In [ ]:
data = train.load_data(SMOKE)
results = []

## Baseline architecture and training

Flatten converts each 28×28 image into 784 input features. Dense(128, ReLU)
learns nonlinear combinations of them. Dense(10, softmax) returns ten digit
probabilities. Adam uses its Keras default learning rate of 0.001. Training uses
batch size 64 for all 50 epochs, without early stopping.

Each run has a context manager so errors close it with FAILED status. TensorFlow
autologging records epoch metrics and model summaries. Explicit metrics and
model artifacts make comparison and retrieval reliable on DagsHub.


In [ ]:
baseline = train.build_model(train.CONFIGS[0])
baseline.summary()
del baseline
results.append(train.run_experiment(train.CONFIGS[0], data, output_dir, SMOKE))

## Experiment 2 — wider hidden layer

Increase the hidden layer to 256 neurons; retain Adam, batch size 64, and 50 epochs.
Greater capacity can improve fitting, but may also increase overfitting.


In [ ]:
results.append(train.run_experiment(train.CONFIGS[1], data, output_dir, SMOKE))

## Experiment 3 — two hidden layers

Use hidden layers of 128 and 64 ReLU neurons with Adam, batch size 64, and 50 epochs.
The second layer can learn further combinations of the first layer's features.


In [ ]:
results.append(train.run_experiment(train.CONFIGS[2], data, output_dir, SMOKE))

## Experiment 4 — SGD and smaller batches

Return to 128 hidden neurons and use SGD (default learning rate 0.01), batch size
32, and 50 epochs. This changes both optimizer and batch size, so differences
cannot be attributed to either factor alone. Actual learning rates are logged.


In [ ]:
results.append(train.run_experiment(train.CONFIGS[3], data, output_dir, SMOKE))

## Compare validation performance, then evaluate the selected model

Compare the final-epoch validation accuracies; break ties with lower validation
loss. The saved model contains final-epoch weights for a fair comparison. There
is no best-epoch checkpoint selection. Only after ranking the runs do we evaluate
the winning model on all 10,000 test images. Smoke mode skips the test evaluation.

Read the generated analysis alongside the learning curves. If training accuracy
continues improving while validation performance stagnates or worsens, discuss
possible overfitting. A single seed cannot prove one architecture always wins.


In [ ]:
comparison = train.finalize_results(results, data, output_dir, tracking, SMOKE)
display(comparison)
display(Image(filename=str(output_dir / 'comparison.png')))
display(Markdown((output_dir / 'analysis.md').read_text(encoding='utf-8')))

## Verify and collect submission evidence

Run the checks below, then follow SUBMISSION_CHECKLIST.md and README.md to capture
the **actual DagsHub interface**: run list, parameters, metrics and a comparison
of at least two runs. Local plots are useful analysis but do not replace these
screenshots. Copy your real GitHub URL, DagsHub URL and tracking URI into your
submission document. Keep tokens and passwords out of screenshots.

If a training cell fails, inspect the error and its FAILED run. Restart the
kernel and rerun from the top to produce a fresh, complete four-run comparison.


In [ ]:
from verify_project import verify_execution
verify_execution(output_dir)
print('Actual tracking URI:', tracking['tracking_uri'])
print('Smoke test only:', SMOKE)